# Feature Selection

> Shared feature scoring and hybrid-union selection library.
>
> Provides `score_features()` and `select_features()` — used by both
> `kreview run` (monolithic) and `kreview select` (standalone CLI).

In [ ]:
#| default_exp selection

In [ ]:
#| export
"""Feature scoring and selection for kreview.

Provides shared functions used by both ``kreview run`` (monolithic) and
``kreview select`` (standalone CLI) to ensure identical feature selection
logic across all execution modes.

Two public functions:

- :func:`score_features` — compute univariate AUC, mutual information,
  and statistical tests (KW, Cohen's d) for every numeric feature column.
- :func:`select_features` — apply mRMR (default) or hybrid-union
  selection with a zero-variance guard.
"""

import numpy as np
import pandas as pd
import structlog

log = structlog.get_logger()


# ---------------------------------------------------------------------------
# Shared helpers
# ---------------------------------------------------------------------------

# Canonical labels used for the binary modeling target.
# Positives: True ctDNA+, Possible ctDNA+
# Negatives: Healthy Normal, Possible ctDNA-, Possible ctDNA−
_MODEL_LABELS = frozenset(
    [
        "True ctDNA+",
        "Possible ctDNA+",
        "Healthy Normal",
        "Possible ctDNA-",
        "Possible ctDNA\u2212",
    ]
)
_POSITIVE_LABELS = frozenset(["True ctDNA+", "Possible ctDNA+"])


def build_binary_target(
    df: pd.DataFrame, label_col: str = "label"
) -> tuple[pd.DataFrame, np.ndarray]:
    """Filter to modelable samples and build a binary target vector.

    Keeps only samples whose label is in the canonical 5-tier model set
    (True ctDNA+, Possible ctDNA+, Healthy Normal, Possible ctDNA-/−).
    Returns a binary target: 1 for positives, 0 for negatives.

    Args:
        df: DataFrame with a ``label`` column.
        label_col: Name of the label column.

    Returns:
        ``(model_df, y)`` where ``model_df`` contains only samples with
        labels in ``_MODEL_LABELS`` and ``y`` is a binary numpy array.

    Raises:
        ValueError: If fewer than 20 samples remain or only one class is
            present (models cannot train).
    """
    mask = df[label_col].isin(_MODEL_LABELS)
    model_df = df[mask].copy()
    y = model_df[label_col].isin(_POSITIVE_LABELS).astype(int).values

    if len(model_df) < 20:
        raise ValueError(
            f"Insufficient samples for scoring: {len(model_df)} "
            f"(need ≥ 20 after filtering to model labels)"
        )
    if len(np.unique(y)) < 2:
        raise ValueError(
            f"Only one class present in {len(model_df)} samples — "
            f"cannot compute AUC or mutual information"
        )

    return model_df, y


def _impute(df: pd.DataFrame, strategy: str) -> pd.DataFrame:
    """Apply imputation strategy to a feature DataFrame.

    Args:
        df: DataFrame with numeric features (may contain NaN).
        strategy: One of ``'zero'``, ``'mean'``, ``'median'``.

    Returns:
        DataFrame with NaN values filled according to strategy.
    """
    if strategy == "mean":
        return df.fillna(df.mean())
    elif strategy == "median":
        return df.fillna(df.median())
    elif strategy != "zero":
        log.warning(
            "unknown_impute_strategy",
            strategy=strategy,
            fallback="zero",
        )
    return df.fillna(0)  # default: zero


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------


def score_features(
    matrix_df: pd.DataFrame,
    *,
    label_col: str = "label",
    cv_folds: int = 5,
    compute_auc: bool = True,
    random_state: int = 42,
) -> pd.DataFrame:
    """Score every numeric feature by statistical tests, univariate AUC, and MI.

    Computes per-feature:

    - **Statistical tests** via :func:`evaluate_feature`: Kruskal-Wallis,
      Cohen's d, group medians, etc.
    - **Univariate AUC** (optional): Cross-validated logistic regression AUC
      per feature.
    - **Mutual information**: Quantifies non-linear dependency between feature
      and binary target.

    Args:
        matrix_df: DataFrame from ``kreview extract`` containing
            ``LABEL_META_COLS`` metadata columns and numeric feature columns.
        label_col: Name of the label column (default ``"label"``).
        cv_folds: Number of folds for univariate AUC cross-validation.
        compute_auc: If ``False``, skip univariate AUC (selection degrades
            to MI-only).

    Returns:
        DataFrame with one row per feature and columns including
        ``feature_column``, ``univariate_auc``, ``mutual_info``, plus all
        columns from :func:`evaluate_feature`.

    Raises:
        ValueError: If no numeric feature columns found, or insufficient
            samples for scoring.
    """
    from kreview.core import LABEL_META_COLS
    from kreview.eval_engine import (
        evaluate_feature,
        mutual_info_score as mi_score,
        univariate_auc,
    )

    # ── Identify numeric feature columns ──
    numeric_cols = [
        c
        for c in matrix_df.select_dtypes(include=np.number).columns
        if c not in LABEL_META_COLS
    ]
    if not numeric_cols:
        raise ValueError(
            "No numeric feature columns found in matrix "
            f"(columns: {list(matrix_df.columns)[:10]}...)"
        )

    log.info("score_features_start", n_features=len(numeric_cols))

    # ── Statistical tests (KW, Cohen's d, etc.) ──
    eval_results = []
    for col in numeric_cols:
        res = evaluate_feature(
            matrix_df[col],
            matrix_df[label_col],
            total_fragments=matrix_df.get("n_total_somatic_snvs"),
            max_vaf=matrix_df.get("max_vaf"),
        )
        res["feature_column"] = col
        eval_results.append(res)

    eval_df = pd.DataFrame(eval_results)

    # ── Build binary target for AUC + MI scoring ──
    model_df, y = build_binary_target(matrix_df, label_col)

    # ── Univariate AUC (optional — CV-based, slower) ──
    if compute_auc:
        log.info("univariate_auc_start", n_features=len(numeric_cols))
        uauc_scores = []
        for col in numeric_cols:
            auc_val = univariate_auc(model_df[col], y, n_folds=cv_folds, random_state=random_state)
            uauc_scores.append(auc_val)
        eval_df["univariate_auc"] = uauc_scores
    else:
        log.warning(
            "univariate_auc_disabled",
            impact="selection_mi_only",
        )

    # ── Mutual Information (always — fast, no CV needed) ──
    log.info("mutual_info_start", n_features=len(numeric_cols))
    mi_scores = []
    for col in numeric_cols:
        mi_val = mi_score(model_df[col], y, random_state=random_state)
        mi_scores.append(mi_val)
    eval_df["mutual_info"] = mi_scores

    log.info(
        "score_features_complete",
        n_features=len(numeric_cols),
        n_auc_above_055=(
            int((eval_df["univariate_auc"] > 0.55).sum())
            if "univariate_auc" in eval_df.columns
            else 0
        ),
        n_mi_above_zero=int((eval_df["mutual_info"] > 0.0).sum()),
    )

    return eval_df


def select_features(
    matrix_df: pd.DataFrame,
    eval_stats: pd.DataFrame,
    *,
    top_percentile: float = 50.0,
    impute_strategy: str = "median",
    label_col: str = "label",
    strategy: str = "mrmr",
) -> tuple[pd.DataFrame, dict]:
    """Apply feature selection and return a filtered matrix.

    Strategies:
    - ``'mrmr'``: Minimum Redundancy Maximum Relevance (redundancy-aware)
    - ``'hybrid_union'``: Top N% AUC ∪ Top N% MI (legacy)

    Selection algorithm (mRMR):

    1. Compute ``n_keep = max(1, n_features × top_percentile / 100)``.
    2. Impute NaNs using ``impute_strategy``.
    3. Apply mRMR to select top ``n_keep`` features.
    4. **Variance guard**: drop zero-variance features.
    5. Return the matrix with only ``LABEL_META_COLS`` + selected features.

    Selection algorithm (hybrid_union):

    1. Compute ``n_keep = max(1, n_features × top_percentile / 100)``.
    2. Take top ``n_keep`` features by ``univariate_auc`` (if present).
    3. Take top ``n_keep`` features by ``mutual_info`` (if present).
    4. **Union** both sets — captures linear (AUC) and non-linear (MI) signal.
    5. **Variance guard**: impute + drop zero-variance features.
    6. Return the matrix with only ``LABEL_META_COLS`` + selected features.

    Args:
        matrix_df: Full matrix DataFrame from ``kreview extract``.
        eval_stats: Feature scoring DataFrame from :func:`score_features`.
        top_percentile: Top N% per metric to keep (default 50).
        impute_strategy: Strategy for NaN imputation before variance check.
        label_col: Name of the label column.
        strategy: Feature selection strategy ('mrmr' or 'hybrid_union').

    Returns:
        Tuple of ``(selected_matrix_df, selection_qc_dict)``:

        - ``selected_matrix_df``: DataFrame with metadata + selected features.
        - ``selection_qc_dict``: Audit trail with counts and method metadata.

    Raises:
        ValueError: If all features are constant after selection (nothing to model).
    """
    from kreview.core import LABEL_META_COLS

    # ── Identify feature columns in the matrix ──
    numeric_cols = [
        c
        for c in matrix_df.select_dtypes(include=np.number).columns
        if c not in LABEL_META_COLS
    ]
    n_keep = max(1, int(len(numeric_cols) * (top_percentile / 100.0)))

    if strategy == "mrmr":
        from mrmr import mrmr_classif

        model_df, y = build_binary_target(matrix_df, label_col)
        X_imputed = _impute(model_df[numeric_cols], impute_strategy)
        y_series = pd.Series(y, index=X_imputed.index, name="target")

        selected_mrmr = mrmr_classif(
            X=X_imputed, y=y_series, K=n_keep, show_progress=False
        )

        # Variance guard
        nonconst = [c for c in selected_mrmr if X_imputed[c].std() > 0]
        n_variance_dropped = len(selected_mrmr) - len(nonconst)

        if not nonconst:
            raise ValueError("All mRMR-selected features are constant (zero variance)")

        selected_features = nonconst
        selection_qc = {
            "method": "mrmr",
            "total_input_features": len(numeric_cols),
            "target_percentile": top_percentile,
            "n_keep_requested": n_keep,
            "n_mrmr_selected": len(selected_mrmr),
            "n_after_variance_guard": len(selected_features),
            "n_variance_dropped": n_variance_dropped,
            "impute_strategy": impute_strategy,
        }

        log.info("feature_selection_complete", **selection_qc)

        meta_cols = [c for c in matrix_df.columns if c in LABEL_META_COLS]
        output_cols = meta_cols + selected_features
        selected_matrix = matrix_df[output_cols].copy()

        log.info(
            "selected_matrix_built",
            n_samples=len(selected_matrix),
            n_meta_cols=len(meta_cols),
            n_feature_cols=len(selected_features),
            n_total_cols=len(output_cols),
        )
        return selected_matrix, selection_qc

    # ── Hybrid union: top N% by AUC ∪ top N% by MI ──
    top_by_auc = set()
    top_by_mi = set()

    if "univariate_auc" in eval_stats.columns:
        top_by_auc = set(
            eval_stats.nlargest(n_keep, "univariate_auc")["feature_column"]
        )
    if "mutual_info" in eval_stats.columns:
        top_by_mi = set(eval_stats.nlargest(n_keep, "mutual_info")["feature_column"])

    union_feats = list(top_by_auc | top_by_mi)

    # Fallback: if both metrics are empty, take first n_keep features
    if not union_feats:
        log.warning(
            "selection_fallback",
            reason="no_scored_features",
            n_keep=n_keep,
        )
        union_feats = numeric_cols[:n_keep]

    # ── Variance guard: drop constant features after imputation ──
    # Filter matrix to model-eligible samples for variance check.
    model_df, _ = build_binary_target(matrix_df, label_col)
    X_imputed = _impute(model_df[union_feats], impute_strategy)
    nonconst = [c for c in union_feats if X_imputed[c].std() > 0]
    n_variance_dropped = len(union_feats) - len(nonconst)

    if n_variance_dropped > 0:
        log.info(
            "variance_guard_dropped",
            n_dropped=n_variance_dropped,
            n_remaining=len(nonconst),
        )

    if not nonconst:
        raise ValueError(
            f"All {len(union_feats)} selected features are constant "
            f"(zero variance) — cannot proceed with modeling"
        )

    selected_features = nonconst

    # ── Build selection QC metadata ──
    selection_qc = {
        "method": "hybrid_union",
        "total_input_features": len(numeric_cols),
        "target_percentile": top_percentile,
        "n_keep_per_metric": n_keep,
        "n_selected_union": len(top_by_auc | top_by_mi),
        "n_after_variance_guard": len(selected_features),
        "n_variance_dropped": n_variance_dropped,
        "n_overlap_both": len(top_by_auc & top_by_mi),
        "n_auc_only": len(top_by_auc - top_by_mi),
        "n_mi_only": len(top_by_mi - top_by_auc),
        "impute_strategy": impute_strategy,
    }

    log.info("feature_selection_complete", **selection_qc)

    # ── Build output matrix: metadata cols + selected features only ──
    meta_cols = [c for c in matrix_df.columns if c in LABEL_META_COLS]
    output_cols = meta_cols + selected_features
    selected_matrix = matrix_df[output_cols].copy()

    log.info(
        "selected_matrix_built",
        n_samples=len(selected_matrix),
        n_meta_cols=len(meta_cols),
        n_feature_cols=len(selected_features),
        n_total_cols=len(output_cols),
    )

    return selected_matrix, selection_qc